COMPSCI 760 Group 9, statistical analysis

Input : merged_3system_with_metrics.csv
Output: <out>/intervals_long.csv   paired bootstrap CIs on gaps + accent spread
        <out>/accent_means_ci.csv  accent means per system with CIs, ranks, spread
        <out>/spearman.csv         WER vs cascade chrF++ / vs Direct-minus-Cascade gap
        <out>/tests.csv            Kruskal-Wallis, Dunn (Holm), effect sizes
        <out>/intervals_unk_free.csv  gap intervals on the clips where Direct did not output <unk>

Usage
  python statistical_analysis.py                   # reads merged_3system_with_metrics.csv in the runs folder

Expected columns (fixed): id, sample_id (speaker), primary_accent, wer_norm,
chrfpp_direct, chrfpp_600m, chrfpp_33b, direct_has_unk

Conventions
  * outcome  = sentence-level chrF++ (0-100)
  * gap      = first system minus second system (negative => first system scores lower)
  * default resampling unit = speaker, stratified by accent; clip-level reported alongside
  * seed 760, 1000 bootstrap iterations, 95% percentile intervals
"""

In [ ]:
%pip install numpy pandas scip

In [ ]:
from __future__ import annotations

import argparse
import itertools
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

In [ ]:
from pathlib import Path

def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, cwd.parent]:
        if (candidate / "data").exists() and (candidate / "runs").exists():
            return candidate
    return cwd

PROJECT_ROOT = find_project_root()

DATA_FILE = PROJECT_ROOT / "runs" / "merged_3system_with_metrics.csv"

OUTPUT_DIR = (
    PROJECT_ROOT
    / "runs"
    / "statistical_analysis_results"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_FILE.is_file():
    raise FileNotFoundError(f"Merged CSV not found: {DATA_FILE}")

print("Project root:", PROJECT_ROOT)
print("Input:", DATA_FILE)
print("Output:", OUTPUT_DIR)

In [ ]:
SEED, N_BOOT = 760, 1000
SYSTEMS = ["direct", "c600m", "c33b"]
LABEL = {"direct": "Direct", "c600m": "Cascade-600M", "c33b": "Cascade-3.3B"}
GAPS = {"direct_minus_600m": ("direct", "c600m"),
        "direct_minus_33b": ("direct", "c33b"),
        "33b_minus_600m": ("c33b", "c600m")}
GAP_LABEL = {"direct_minus_600m": "Direct minus Cascade-600M",
             "direct_minus_33b": "Direct minus Cascade-3.3B",
             "33b_minus_600m": "Cascade-3.3B minus Cascade-600M"}
PROVIDED_GAP_COLS = {"direct_minus_600m": "gap_direct_600m",
                     "direct_minus_33b": "gap_direct_33b",
                     "33b_minus_600m": "gap_33b_600m"}
OVERALL = "overall"


# ----------------------------------------------------------------------------
# 1. Loading (columns are fixed to merged_3system_with_metrics.csv)
# ----------------------------------------------------------------------------


In [ ]:
COLS = {"id": "id", "speaker": "sample_id", "accent": "primary_accent", "wer": "wer_norm",
        "direct": "chrfpp_direct", "c600m": "chrfpp_600m", "c33b": "chrfpp_33b",
        "unk": "direct_has_unk"}

def read_csv(path):
    if not Path(path).exists():
        sys.exit(f"File not found: {path}\nPut the CSV in the folder you run the command from, or pass --data <path>.")
    return pd.read_csv(path, encoding="utf-8-sig")   # utf-8-sig also copes with a BOM


def prepare(raw):
    missing = [c for c in COLS.values() if c not in raw.columns]
    if missing:
        sys.exit(f"Missing expected column(s): {missing}\nColumns in the file: {list(raw.columns)}")
    if raw[COLS["speaker"]].isna().any():
        sys.exit(f"{int(raw[COLS['speaker']].isna().sum())} rows have an empty {COLS['speaker']}; fix the file first.")
    d = pd.DataFrame({
        "id": raw[COLS["id"]].astype(str),
        "accent": raw[COLS["accent"]].astype(str).str.strip(),
        "speaker": raw[COLS["speaker"]].astype(str),
        "wer": pd.to_numeric(raw[COLS["wer"]], errors="coerce"),
        "unk": raw[COLS["unk"]].astype(str).str.strip().str.lower().isin(["true", "1", "1.0", "yes"]),
    })
    for s in SYSTEMS:
        d[s] = pd.to_numeric(raw[COLS[s]], errors="coerce")
    for g, (a, b) in GAPS.items():
        d[g] = d[a] - d[b]
        pc = PROVIDED_GAP_COLS[g]
        if pc in raw.columns:
            diff = (pd.to_numeric(raw[pc], errors="coerce") - d[g]).abs().max()
            if diff > 1e-6:
                print(f"  WARNING: provided column {pc} differs from {a}-{b} "
                      f"(max abs diff {diff:.4g}). Using recomputed values (first minus second).")
    n0 = len(d)
    d = d.dropna(subset=SYSTEMS).reset_index(drop=True)
    if len(d) < n0:
        print(f"  WARNING: dropped {n0 - len(d)} rows with missing chrF++.")
    if d["id"].duplicated().any():
        sys.exit("Duplicate ids in the merged file - fix the merge before analysing.")
    multi = d.groupby("speaker")["accent"].nunique()
    if (multi > 1).any():
        print(f"  WARNING: {(multi > 1).sum()} speaker ids appear under more than one accent; "
              "they are treated as separate clusters within each accent.")
    return d



# ----------------------------------------------------------------------------
# 2. Cluster bootstrap (stratified by accent, one shared set of resamples)
# ----------------------------------------------------------------------------


In [ ]:
class StratifiedBoot:
    """Resample clusters (speakers, or single clips) with replacement WITHIN each accent.
    The same resampled clusters are used for every value column, so every statistic
    (system means, paired gaps, spreads) is paired across systems."""

    def __init__(self, df, value_cols, unit, B, seed):
        self.cols, self.accents = list(value_cols), sorted(df["accent"].unique())
        rng = np.random.default_rng(seed)
        self.num, self.den = {}, {}
        for a in self.accents:
            d = df[df["accent"] == a]
            key = d["speaker"].to_numpy() if unit == "speaker" else np.arange(len(d))
            codes, _ = pd.factorize(key)
            G = codes.max() + 1
            S = np.zeros((G, len(self.cols)))
            np.add.at(S, codes, d[self.cols].to_numpy())
            n = np.bincount(codes, minlength=G).astype(float)
            idx = rng.integers(0, G, size=(B, G))
            self.num[a] = S[idx].sum(axis=1)          # (B, K) resampled sums
            self.den[a] = n[idx].sum(axis=1)          # (B,)   resampled clip counts

    def means(self, accent=None):                      # -> (B, K)
        if accent is None:
            return sum(self.num.values()) / sum(self.den.values())[:, None]
        return self.num[accent] / self.den[accent][:, None]

    def spread(self):                                  # max - min of accent means, (B, K)
        M = np.stack([self.means(a) for a in self.accents])
        return M.max(axis=0) - M.min(axis=0)


def _ci(x):
    lo, hi = np.nanpercentile(x, [2.5, 97.5])
    return lo, hi, np.nanstd(x, ddof=1)


def _row(statistic, name, stratum, unit, d, est, draws, **extra):
    lo, hi, se = _ci(draws)
    return dict(statistic=statistic, comparison=name, stratum=stratum, unit=unit,
                n_clips=len(d), n_speakers=d["speaker"].nunique(), estimate=est,
                ci_low=lo, ci_high=hi, boot_se=se,
                excludes_zero=(bool(lo > 0 or hi < 0)
                               if statistic in ("mean_gap", "spread_difference") else np.nan), **extra)


def bootstrap_tables(df, unit, B, seed):
    cols = SYSTEMS + list(GAPS)
    bt = StratifiedBoot(df, cols, unit, B, seed)
    k = {c: i for i, c in enumerate(cols)}
    iv, am = [], []
    for stratum in [OVERALL] + bt.accents:
        d = df if stratum == OVERALL else df[df["accent"] == stratum]
        M = bt.means(None if stratum == OVERALL else stratum)
        obs = d[cols].mean()
        for g in GAPS:
            iv.append(_row("mean_gap", GAP_LABEL[g], stratum, unit, d, obs[g], M[:, k[g]]))
        if stratum != OVERALL:
            for s in SYSTEMS:
                r = _row("accent_mean", LABEL[s], stratum, unit, d, obs[s], M[:, k[s]])
                r["system"] = r.pop("comparison")
                am.append(r)
    # between-accent spread (max - min of the 7 accent means)
    am_df = pd.DataFrame(am)
    SP = bt.spread()
    obs_means = df.groupby("accent")[cols].mean()
    spread_obs = obs_means.max() - obs_means.min()
    for s in SYSTEMS:
        hi_a, lo_a = obs_means[s].idxmax(), obs_means[s].idxmin()
        r = _row("spread_max_minus_min", f"spread_{LABEL[s]}", f"{len(bt.accents)} accents",
                 unit, df, spread_obs[s], SP[:, k[s]])
        iv.append(r)
        r2 = dict(r, system=LABEL[s], stratum=f"max: {hi_a} / min: {lo_a}")
        r2.pop("comparison")
        am_df = pd.concat([am_df, pd.DataFrame([r2])], ignore_index=True)
    for s1, s2 in [("direct", "c600m"), ("direct", "c33b"), ("c33b", "c600m")]:
        r = _row("spread_difference", f"spread {LABEL[s1]} minus {LABEL[s2]}",
                 f"{len(bt.accents)} accents", unit, df, spread_obs[s1] - spread_obs[s2],
                 SP[:, k[s1]] - SP[:, k[s2]])
        am_df = pd.concat([am_df, pd.DataFrame([dict(r, system=r.pop("comparison"))])],
                          ignore_index=True)
    # rank of each accent within each system (1 = highest mean)
    m = am_df["statistic"] == "accent_mean"
    am_df["rank_within_system"] = np.nan
    am_df.loc[m, "rank_within_system"] = (am_df[m].groupby(["system"])["estimate"]
                                          .rank(ascending=False, method="min"))
    am_df = am_df.rename(columns={"stratum": "accent"})
    am_df = am_df[["statistic", "system", "accent", "unit"] +
                  [c for c in am_df.columns if c not in ("statistic", "system", "accent", "unit")]]
    return pd.DataFrame(iv), am_df



# ----------------------------------------------------------------------------
# 3. Spearman with speaker-cluster (or clip) bootstrap CI
# ----------------------------------------------------------------------------


In [ ]:
class Clusters:
    """Fast row-index resampler: speakers (or clips) with replacement within each accent."""

    def __init__(self, d, unit):
        self.parts = []
        for _, da in d.groupby("accent", sort=True):
            pos = da.index.to_numpy()
            key = da["speaker"].to_numpy() if unit == "speaker" else pos
            codes, _ = pd.factorize(key)
            order = np.argsort(codes, kind="stable")
            lens = np.bincount(codes)
            self.parts.append((pos[order], np.cumsum(lens) - lens, lens))

    def draw(self, rng):
        out = []
        for ordpos, starts, lens in self.parts:
            G = len(lens)
            sp = rng.integers(0, G, G)
            l, s = lens[sp], starts[sp]
            offs = np.repeat(s - (np.cumsum(l) - l), l) + np.arange(l.sum())
            out.append(ordpos[offs])
        return np.concatenate(out)


def _spearman_ranked(rx, y):
    ry = stats.rankdata(y)
    rx_c, ry_c = rx - rx.mean(), ry - ry.mean()
    den = np.sqrt((rx_c ** 2).sum() * (ry_c ** 2).sum())
    return (rx_c * ry_c).sum() / den if den > 0 else np.nan


def spearman_table(df, unit, B, seed):
    df = df.dropna(subset=["wer"]).reset_index(drop=True)
    ys = [("c600m", "Cascade-600M sentence chrF++"), ("c33b", "Cascade-3.3B sentence chrF++"),
          ("direct_minus_600m", "gap Direct minus Cascade-600M"),
          ("direct_minus_33b", "gap Direct minus Cascade-3.3B")]
    wer = df["wer"].to_numpy()
    rows = []
    for stratum in [OVERALL] + sorted(df["accent"].unique()):
        d = df if stratum == OVERALL else df[df["accent"] == stratum]
        cl, rng = Clusters(d, unit), np.random.default_rng(seed)
        boots = np.empty((B, len(ys)))
        for b in range(B):
            rows_b = cl.draw(rng)
            rx = stats.rankdata(wer[rows_b])
            for j, (y, _) in enumerate(ys):
                boots[b, j] = _spearman_ranked(rx, df[y].to_numpy()[rows_b])
        for j, (y, lab) in enumerate(ys):
            rho = stats.spearmanr(d["wer"], d[y])[0]
            lo, hi = np.nanpercentile(boots[:, j], [2.5, 97.5])
            rows.append(dict(x="normalised WER", y=lab, stratum=stratum, unit=unit,
                             n_clips=len(d), n_speakers=d["speaker"].nunique(),
                             rho=rho, ci_low=lo, ci_high=hi,
                             excludes_zero=bool(lo > 0 or hi < 0)))
    return pd.DataFrame(rows)




# ----------------------------------------------------------------------------
# 4. Kruskal-Wallis, Dunn (Holm), effect sizes
# ----------------------------------------------------------------------------


In [ ]:
def holm(p):
    p = np.asarray(p, float)
    m, order, adj, run = len(p), np.argsort(p), np.empty(len(p)), 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i])
        adj[i] = min(1.0, run)
    return adj


def cliffs_delta(a, b):
    """P(a>b) - P(a<b). Identical to the Mann-Whitney rank-biserial correlation."""
    r = stats.rankdata(np.concatenate([a, b]))
    u1 = r[:len(a)].sum() - len(a) * (len(a) + 1) / 2
    return 2 * u1 / (len(a) * len(b)) - 1


def paired_rank_biserial(x):
    x = np.asarray(x, float)
    x = x[x != 0]
    if len(x) == 0:
        return np.nan
    r = stats.rankdata(np.abs(x))
    wp, wm = r[x > 0].sum(), r[x < 0].sum()
    return (wp - wm) / (wp + wm)


def dunn(groups):
    names = list(groups)
    allv = np.concatenate([groups[n] for n in names])
    N = len(allv)
    ranks = stats.rankdata(allv)
    _, t = np.unique(allv, return_counts=True)
    sigma2 = N * (N + 1) / 12 - (t ** 3 - t).sum() / (12 * (N - 1))
    mr, n_i, start = {}, {}, 0
    for n in names:
        L = len(groups[n])
        mr[n], n_i[n] = ranks[start:start + L].mean(), L
        start += L
    out = []
    for a, b in itertools.combinations(names, 2):
        z = (mr[a] - mr[b]) / np.sqrt(sigma2 * (1 / n_i[a] + 1 / n_i[b]))
        out.append(dict(a=a, b=b, z=z, p=2 * stats.norm.sf(abs(z)),
                        delta=cliffs_delta(groups[a], groups[b]), n_a=n_i[a], n_b=n_i[b]))
    for r, ph in zip(out, holm([r["p"] for r in out])):
        r["p_holm"] = ph
    return out


def tests_table(df, accent_means):
    spk = df.groupby(["accent", "speaker"], as_index=False)[SYSTEMS + list(GAPS)].mean()
    rows = []
    targets = [(g, GAP_LABEL[g]) for g in GAPS] + [(s, f"{LABEL[s]} sentence chrF++") for s in SYSTEMS]
    for level, data in [("clip", df), ("speaker", spk)]:
        for col, lab in targets:
            groups = {a: x[col].to_numpy() for a, x in data.groupby("accent")}
            H, p = stats.kruskal(*groups.values())
            N = sum(len(v) for v in groups.values())
            rows.append(dict(test="kruskal_wallis", target=lab, level=level, stratum_a="all accents",
                             stratum_b="", n_a=N, n_b=np.nan, statistic=H, df=len(groups) - 1, p=p,
                             p_holm=np.nan, effect_name="epsilon_squared",
                             effect_size=H * (N + 1) / (N ** 2 - 1)))
            if col in GAPS:
                for r in dunn(groups):
                    rows.append(dict(test="dunn_holm", target=lab, level=level, stratum_a=r["a"],
                                     stratum_b=r["b"], n_a=r["n_a"], n_b=r["n_b"], statistic=r["z"],
                                     df=np.nan, p=r["p"], p_holm=r["p_holm"],
                                     effect_name="cliffs_delta(a vs b)", effect_size=r["delta"]))
                for stratum, x in [(OVERALL, data)] + list(data.groupby("accent")):
                    v = x[col].to_numpy()
                    try:
                        W, pw = stats.wilcoxon(v[v != 0])
                    except ValueError:
                        W, pw = np.nan, np.nan
                    rows.append(dict(test="wilcoxon_signed_rank", target=lab, level=level,
                                     stratum_a=stratum, stratum_b="", n_a=len(v), n_b=np.nan,
                                     statistic=W, df=np.nan, p=pw, p_holm=np.nan,
                                     effect_name="rank_biserial_paired", effect_size=paired_rank_biserial(v)))
    # does the accent ordering agree across systems?  (7 accent means per system)
    m = accent_means[(accent_means["statistic"] == "accent_mean") & (accent_means["unit"] == "speaker")]
    piv = m.pivot(index="accent", columns="system", values="estimate")
    for s1, s2 in itertools.combinations(SYSTEMS, 2):
        rho = stats.spearmanr(piv[LABEL[s1]], piv[LABEL[s2]])[0]
        same = bool((piv[LABEL[s1]].rank() == piv[LABEL[s2]].rank()).all())
        rows.append(dict(test="accent_order_spearman", target=f"{LABEL[s1]} vs {LABEL[s2]}", level="accent means",
                         stratum_a="7 accents", stratum_b="identical order" if same else "order differs",
                         n_a=len(piv), n_b=np.nan, statistic=rho, df=np.nan, p=np.nan, p_holm=np.nan,
                         effect_name="spearman_rho", effect_size=rho))
    return pd.DataFrame(rows)



# ----------------------------------------------------------------------------
# 5. main
# ----------------------------------------------------------------------------


In [ ]:
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", default=DATA_FILE, help=f"path to the merged CSV (default merged_3system_with_metrics.csv)")
    ap.add_argument("--out", default=OUTPUT_DIR, help="output folder (default statistical_analysis_results)")
    ap.add_argument("--B", type=int, default=N_BOOT, help="bootstrap iterations (default 1000)")
    ap.add_argument("--seed", type=int, default=SEED, help="random seed (default 760)")
    args = ap.parse_args()


    raw = read_csv(args.data)
    df = prepare(raw)

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    iv_s, am_s = bootstrap_tables(df, "speaker", args.B, args.seed)
    iv_c, am_c = bootstrap_tables(df, "clip", args.B, args.seed)
    iv, am = pd.concat([iv_s, iv_c], ignore_index=True), pd.concat([am_s, am_c], ignore_index=True)
    print(f"\nbootstrap CIs done ({time.time() - t0:.0f}s)")
    sp = pd.concat([spearman_table(df, u, args.B, args.seed) for u in ("speaker", "clip")], ignore_index=True)
    print(f"spearman done ({time.time() - t0:.0f}s)")
    tests = tests_table(df, am)
    print(f"tests done ({time.time() - t0:.0f}s)")

    iv.to_csv(out / "intervals_long.csv", index=False)
    am.to_csv(out / "accent_means_ci.csv", index=False)
    sp.to_csv(out / "spearman.csv", index=False)
    tests.to_csv(out / "tests.csv", index=False)
    free = df[~df["unk"]].reset_index(drop=True)
    ivf, _ = bootstrap_tables(free, "speaker", args.B, args.seed)
    ivf = ivf[ivf["comparison"].str.contains("Direct")].assign(subset="unk-free clips", n_unk_dropped=int(df["unk"].sum()))
    ivf.to_csv(out / "intervals_unk_free.csv", index=False)
    print(f"unk sensitivity: {int(df['unk'].sum())} Direct <unk> clips excluded from all systems")

    print("\nOverall gaps (speaker-clustered 95% CI):")
    show = iv[(iv["unit"] == "speaker") & (iv["stratum"] == OVERALL)]
    for _, r in show.iterrows():
        print(f"  {r['comparison']:34s} {r['estimate']:7.2f}  [{r['ci_low']:.2f}, {r['ci_high']:.2f}]")
    print(f"\nWrote files to {out.resolve()}")

In [ ]:
sys.argv = ["statistical_analysis.py"]
main()